# 03 · 베이스라인 학습과 성능 지표

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-axis/Wind-Power-Forecasting/blob/main/notebooks/03_baseline_training_and_metrics.ipynb)

현재 구현된 14개 베이스라인과 EMFN, 총 15개 모델 계열을 동일 평가 시점에 재실험합니다. 6개 시계(1/3/6/9/12/24h), 학습 시드 42/2026/3407을 사용합니다. Persistence와 ARIMA는 시드 반복을 만들지 않습니다. 모델별 입력 예산과 출력 형태는 아래 표로 구분합니다. 과거 v1/v2와 구조 어블레이션은 이 베이스라인 목록에 포함하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys

# Google Colab 자동 감지 및 환경 설정
if 'google.colab' in sys.modules or (os.path.exists('/content') and not (Path.cwd() / "configs/base_config.yaml").exists()):
    repo_dir = Path('/content/Wind-Power-Forecasting')
    if not repo_dir.exists():
        print("Google Colab 환경 감지: Wind-Power-Forecasting 저장소를 클론합니다...")
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lab-axis/Wind-Power-Forecasting.git", str(repo_dir)], check=True)
        print("필수 의존 패키지를 설치합니다 (colab/requirements-colab.txt)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "colab/requirements-colab.txt")], check=True)
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    try:
        get_ipython().run_line_magic('cd', str(repo_dir))
    except Exception:
        pass

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/base_config.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
runtime = ROOT / ".experiment_archive/python_runtime"
if runtime.exists(): sys.path.insert(0, str(runtime))
from IPython.display import display
from src.workflows import research as wf
%matplotlib inline

In [ ]:
wf.model_catalog()

## 1. 고정된 학습 규약

설정: `configs/all_baselines.yaml`, 실행: `src/experiments/run_all_baselines.py`, 생성자: `src/models/benchmark_registry.py`. 신경망 체크포인트와 트리 반복 수, ARIMA 차수는 검증 CRPS로 선택합니다. 결정론 CRPS는 clipped MAE입니다. 학습 손실은 EMFN NLL+Huber, 다른 신경망 Huber, 점예측 LightGBM L2를 유지합니다.

GRU의 마지막 ReLU는 제거해 음수 초기 출력에서도 학습 기울기가 흐르도록 수정했습니다. 새 결과는 수정 전 0 수렴 실행과 구분합니다. 장기 lag LightGBM은 NaN을 자체 처리하고 최대 168h lag·달력 특징을 사용하므로 24h 입력 모델과 구조만의 비교가 아닙니다. Prophet은 기존 연간/주간/일간 계절성 설정의 train-only 달력 예측이며 rolling refit을 하지 않습니다. 학습 자료가 2년 미만이므로 연간 계절성 식별의 한계도 있습니다.

In [ ]:
wf.run_status()

## 2. 재학습 실행 (선택)

기본값 False는 검증된 저장 결과를 읽는 모드입니다. True이면 **EMFN을 포함한 전체 행렬을 처음부터** 새 run 폴더에 학습하며 상당한 시간이 걸립니다. 234는 모델×시계×시드 조합 수이지 독립 적합 함수 호출 수가 아닙니다. ARIMA 후보 5개와 시드별 Prophet은 한 번 적합한 파라미터를 여러 시계에서 사용합니다. 완료 후 검증기를 실행해야 최신 manifest와 보고 표로 승격됩니다.

In [ ]:
RUN_FULL_RETRAINING = False
if RUN_FULL_RETRAINING:
    new_run = wf.train_all()
    wf.publish_run(new_run)

## 3. 검증에 의한 선택과 학습 실패 진단

아래 표의 검증 예측 표준편차가 0이면 상수 예측 경고입니다. 경고를 숨기거나 2025 성능을 보고 좋은 시드만 고르지 않습니다. 선택 없는 Persistence/Prophet은 그 사실을 그대로 기록합니다.

In [ ]:
wf.validation_diagnostics()

## 4. 베이스라인 결과

CRPS ↓, 영발전 Brier ↓, RMSE ↓, MAE ↓, 영발전 AUPRC ↑ 순서입니다. 확률 모델의 RMSE는 예측 평균, MAE는 예측 중앙값입니다. 점예측 모델의 CRPS는 MAE이고 확률 지표는 N/A입니다. 표시한 ±는 시드별 지표의 표본 표준편차이며 앙상블 점수가 아닙니다. 단일 결정론 실행의 SD는 계산하지 않습니다.

In [ ]:
baselines = [name for name in wf.ALL_MODELS if name != "EMFN (Proposed)"]
wf.display_scores(models=baselines, horizons=(1, 6, 24))

모든 시계·시드·구현 지표를 읽으려면 아래 DataFrame을 사용합니다. 파일: `reports/tables/all_baselines_metrics.csv`(468행), `all_baselines_summary.csv`(180행). Notebook의 접힌 표시가 결과 누락을 뜻하지 않습니다.

In [ ]:
wf.metric_definitions()

In [ ]:
all_baseline_metrics = wf.metrics_table(models=baselines, horizons=None, full=True)
display(all_baseline_metrics)
print("Available metric columns:", all_baseline_metrics.columns.tolist())

**다음:** [04 · EMFN 구조와 학습](04_emfn_architecture_and_results.ipynb).